In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


<h1>For dummy submission initially</h1>

In [2]:
import pandas as pd

data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
data.to_csv("submission.csv", index = False)

<h1>Importing all dependencies required:</h1>

In [3]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

<h1>The following is for Milestone 1</h1>
<hr>
<h3>Question 1:</h3> Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [4]:
counts = df['answer'].value_counts()
sum_most_and_least = counts.max() + counts.min()
sum_most_and_least

814

<h3>Question 2:</h3> After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [5]:
def clean_text(text):
    text = text.lower()
    return text.translate(str.maketrans('', '', string.punctuation))

cleaned_prompts = df['prompt'].apply(clean_text)
vocab = set(' '.join(cleaned_prompts).split())
len(vocab)

859

<h3>Question 3:</h3> Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [6]:
row1_tokens = clean_text(df.iloc[0]['prompt']).split()
filtered_tokens = [w for w in row1_tokens if w not in ENGLISH_STOP_WORDS]
len(filtered_tokens)

13

<h3>Question 4:</h3> Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [7]:
combined_text = df['prompt'] + " " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(combined_text)
len(vectorizer.get_feature_names_out())

2762

<h3>Question 5:</h3> Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [8]:
prompt_vec = vectorizer.transform([df.iloc[0]['prompt']])
option_a_vec = vectorizer.transform([df.iloc[0]['A']])
round(cosine_similarity(prompt_vec, option_a_vec)[0][0], 4)

np.float64(0.272)

<h3>Question 6:</h3> Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [9]:
def get_best_option(row):
    options = ['A', 'B', 'C', 'D', 'E']
    prompt_v = vectorizer.transform([row['prompt']])
    scores = {opt: cosine_similarity(prompt_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return max(scores, key=scores.get)

matches = df.apply(lambda row: get_best_option(row) == row['answer'], axis=1)
matches.mean() * 100

np.float64(13.55)

<h3>Question 7:</h3> If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [10]:
gt_7 = 'C'
pred_7 = ['C', 'A', 'B']

score_7 = 1 / (pred_7.index(gt_7) + 1) if gt_7 in pred_7[:3] else 0.0

score_7

1.0

<h3>Question 8:</h3> The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [11]:
gt_8 = 'B'
pred_8 = ['D', 'B', 'E']

score_8 = 1 / (pred_8.index(gt_8) + 1) if gt_8 in pred_8[:3] else 0.0
score_8

0.5

<h3>Question 9:</h3> The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [12]:
top_3_answers = df['answer'].value_counts().index[:3].tolist()
baseline_map3 = df['answer'].apply(
    lambda ans: 1.0 if ans == top_3_answers[0] else 
               (0.5 if ans == top_3_answers[1] else 
               (1/3 if ans == top_3_answers[2] else 0.0))
).mean()
baseline_map3

np.float64(0.42125)

<h3>Question 10:</h3> The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [13]:
def map_at_3(truth, predictions):
    if truth in predictions[:3]:
        rank = predictions.index(truth) + 1
        return 1 / rank
    return 0


def get_top3_preds(row):
    options = ['A', 'B', 'C', 'D', 'E']
    p_v = vectorizer.transform([row['prompt']])
    s = {opt: cosine_similarity(p_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return sorted(s, key=s.get, reverse=True)

all_scores = df.apply(lambda row: map_at_3(row['answer'], get_top3_preds(row)), axis=1)
all_scores.mean()

np.float64(0.2961666666666667)

<h1> MILESTONE 2:</h1>
<HR>


In [14]:
from datasets import load_dataset
from transformers import AutoTokenizer

file_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
dataset = load_dataset('csv', data_files=file_path)['train']

Generating train split: 0 examples [00:00, ? examples/s]

<h3>Question 1 - intro to hugging face transformers and models</h3> 
Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [15]:
def combine_text(example):
    example['combined_text'] = str(example['prompt']) + " " + str(example['A'])
    return example

dataset = dataset.map(combine_text)
combined_text_51 = dataset[51]['combined_text']
len(combined_text_51)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614

<h3>Question 2 - </h3>
Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [16]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokenizer.vocab_size

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522

<h3> Question 3 - </h3>
Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [17]:
#exact ID for the SEP token:
tokenizer.sep_token_id

102

<h3> Question 4 - </h3>
Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [18]:
prompts_list = [str(text) for text in dataset['prompt']]

tokens = tokenizer(prompts_list, padding='max_length', truncation=True, max_length=128, 
    return_tensors='pt')
tokens['input_ids'].shape


torch.Size([2000, 128])

<h3>Question 5 - BERT/RoBERTa architecture and attention mechanisms</h3>
A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  


In [19]:
import torch
from transformers import AutoModel

hidden_size = 768
attention_heads = 12
head_dim = hidden_size / attention_heads
print(int(head_dim))

64


<h3>Question 6 - </h3>
Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [20]:
model = AutoModel.from_pretrained('bert-base-uncased')
inputs_row_0 = tokenizer(dataset[0]['prompt'], return_tensors='pt')

with torch.no_grad():
    outputs_row_0 = model(**inputs_row_0)
outputs_row_0.last_hidden_state.shape

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])

<h3>Question 7 - </h3>
Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [21]:
cls_vector = outputs_row_0.last_hidden_state[0, 0, :] # Batch 0, Token 0
sum_first_5 = cls_vector[:5].sum().item()
round(sum_first_5, 4)

-1.2001

<h3>Question 8 - </h3>
Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [22]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
test_string = "Light-ion fusion is a technique."
inputs_attn = tokenizer(test_string, return_tensors='pt')

with torch.no_grad():
    outputs_attn = model_attn(**inputs_attn)

tokens_list = tokenizer.convert_ids_to_tokens(inputs_attn['input_ids'][0])
fusion_index = tokens_list.index('fusion')

attention_matrix = outputs_attn.attentions[-1]
attention_weight = attention_matrix[0, 0, 0, fusion_index].item()
print(f"Q8 - Attention weight paid to 'fusion': {round(attention_weight, 4)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q8 - Attention weight paid to 'fusion': 0.1025


<h3>Question 9 - context aware embeddings questions</h3>
Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.


In [23]:
from sentence_transformers import SentenceTransformer, util

model_st = SentenceTransformer('all-MiniLM-L6-v2')

prompt_emb = model_st.encode(dataset[0]['prompt'])
option_b_emb = model_st.encode(dataset[0]['B'])

similarity = util.cos_sim(prompt_emb, option_b_emb).item()
round(similarity, 4)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658

<h3>Question 10 - </h3>
Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [24]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util

sent_transformer_model = SentenceTransformer("all-MiniLM-L6-v2")

minilm_map3_score = []
specific_difference_count = 0
option_cols =['A', 'B', 'C', 'D', 'E']

for item in dataset:
    prompt = str(item['prompt'])
    options = [str(item[opt]) for opt in option_cols]
    correct_ans = item['answer']

    #the tf-idf pipeline
    vectorizer = TfidfVectorizer()
    docs = [prompt] + options
    tfidf_matrix = vectorizer.fit_transform(docs)

    tfidf_sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    tfidf_top3_index = tfidf_sims.argsort()[-3:][::-1]
    tfidf_top3_pred = [option_cols[i] for i in tfidf_top3_index]

    #second pipeline for allMiniLM-L6-v2
    prompt_embed = model_st.encode(prompt, convert_to_tensor = True)
    options_embed = model_st.encode(options, convert_to_tensor = True)
    minilm_sims = util.cos_sim(prompt_embed, options_embed)[0].cpu().numpy()

    minilm_top3_index = minilm_sims.argsort()[-3:][::-1]
    minilm_top3_pred = [option_cols[i] for i in minilm_top3_index]

    #comparing
    if correct_ans in minilm_top3_pred:
        rank = minilm_top3_pred.index(correct_ans) + 1
        minilm_map3_score.append(1.0/rank)
    else:
        minilm_map3_score.append(0.0)

    if (correct_ans not in tfidf_top3_pred) and (correct_ans in minilm_top3_pred):
        specific_difference_count += 1


#final calculation
final_map3_score = np.mean(minilm_map3_score)
print(f"Final map@3 score of the all-MiniLM-L6-v2 pipline is: {final_map3_score:.4f}")
print(f"Count of questions meeting the conditions: {specific_difference_count}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Final map@3 score of the all-MiniLM-L6-v2 pipline is: 0.4231
Count of questions meeting the conditions: 564


In [25]:
574

574

<h3>Questions 11 - zero shot classification starts here</h3>
Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [26]:
from transformers import pipeline

#setting up zero-shot classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

prompt_idx_1 = dataset[1]['prompt']
labels_idx_1 = [str(dataset[1]['A']), str(dataset[1]['B']), str(dataset[1]['C'])]

# ques11, zero-shot classification , softmax- probability sum to 1
result_softmax = classifier(prompt_idx_1, candidate_labels=labels_idx_1)
top_score = result_softmax['scores'][0]
round(top_score, 4)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.4575

<h3>Question 12 - </h3>
Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [27]:
result_sigmoid = classifier(prompt_idx_1, candidate_labels=labels_idx_1, multi_label=True)

sum_softmax = sum(result_softmax['scores'])
sum_sigmoid = sum(result_sigmoid['scores'])
absolute_difference = abs(sum_softmax - sum_sigmoid)

round(absolute_difference, 4)

0.9995

<h3>Question 13 - </h3>
Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt_str = f"Question: {dataset[0]['prompt']}. Is the correct answer A: {dataset[0]['A']} or B: {dataset[0]['B']}? Answer with just the letter A or B."

input_ids = tokenizer_t5(prompt_str, return_tensors="pt").input_ids
generated_tokens = model_t5.generate(input_ids, max_new_tokens=5)

exact_output = tokenizer_t5.decode(generated_tokens[0], skip_special_tokens=True)

exact_output

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

'B'